Notes on usage:

- Make sure to [change runtime to GPU](https://www.tutorialspoint.com/google_colab/google_colab_using_free_gpu.htm).
- The transcript will be saved in Files, which you can find in the menu on the left.
- Change the number of speakers below if different from two.
- Pick a bigger model if you want more accuracy and a smaller model if you want the program to run faster ([more info](https://github.com/openai/whisper#available-models-and-languages)).
- If you know the language being spoken is English, then change language to 'English' as this improves performance.


High level overview of what's happening here:


1.   I'm using Open AI's Whisper model to seperate audio into segments and generate transcripts.
2.   I'm then generating speaker embeddings for each segments.
3.   Then I'm using agglomerative clustering on the embeddings to identify the speaker for each segment.   

Let me know if I can make it better!


In [1]:
# upload audio file
from google.colab import files
uploaded = files.upload()
path = next(iter(uploaded))

Saving VocesVarias.wav to VocesVarias.wav


In [13]:
num_speakers = 3 #@param {type:"integer"}

language = 'any' #@param ['any', 'English']

model_size = 'large' #@param ['tiny', 'base', 'small', 'medium', 'large']


model_name = model_size
if language == 'English' and model_size != 'large':
  model_name += '.en'


In [14]:
!pip install -q git+https://github.com/openai/whisper.git > /dev/null
!pip install -q git+https://github.com/pyannote/pyannote-audio > /dev/null

import whisper
import datetime

import subprocess

import torch
import pyannote.audio
from pyannote.audio.pipelines.speaker_verification import PretrainedSpeakerEmbedding
embedding_model = PretrainedSpeakerEmbedding(
    "speechbrain/spkrec-ecapa-voxceleb",
    device=torch.device("cuda"))

from pyannote.audio import Audio
from pyannote.core import Segment

import wave
import contextlib

from sklearn.cluster import AgglomerativeClustering
import numpy as np

/usr/local/lib/python3.11/dist-packages/speechbrain/utils/checkpoints.py:200: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device

In [15]:
if path[-3:] != 'wav':
  subprocess.call(['ffmpeg', '-i', path, 'audio.wav', '-y'])
  path = 'audio.wav'

In [16]:
model = whisper.load_model(model_size)

100%|██████████████████████████████████████| 2.88G/2.88G [00:29<00:00, 105MiB/s]
/usr/local/lib/python3.11/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this exper

In [17]:
result = model.transcribe(path)
segments = result["segments"]

In [18]:
import subprocess

# Asegúrate de que, independientemente del formato original,
# el archivo se convierta a un formato WAV PCM estándar
subprocess.call(['ffmpeg', '-i', path, '-acodec', 'pcm_s16le', 'converted_audio.wav', '-y'])
path = 'converted_audio.wav'  # Actualiza la ruta al nuevo archivo

# Ahora intenta obtener la duración
with contextlib.closing(wave.open(path, 'r')) as f:
    frames = f.getnframes()
    rate = f.getframerate()
    duration = frames / float(rate)

In [19]:
audio = Audio()

def segment_embedding(segment):
  start = segment["start"]
  # Whisper overshoots the end timestamp in the last segment
  end = min(duration, segment["end"])
  clip = Segment(start, end)
  waveform, sample_rate = audio.crop(path, clip)
  return embedding_model(waveform[None])

In [20]:
def segment_embedding(segment):
    start = segment["start"]
    end = min(duration, segment["end"])
    clip = Segment(start, end)

    # Obtener la forma de onda para este segmento
    waveform, sample_rate = audio.crop(path, clip)

    # Verificar y ajustar la forma del tensor
    if not torch.is_tensor(waveform):
        waveform = torch.tensor(waveform)

    # Imprimir la forma actual para depurar
    print(f"Forma original: {waveform.shape}")

    # Transformar a la forma esperada [batch_size=1, channels=1, samples]
    # Si es un tensor 1D (solo muestras)
    if len(waveform.shape) == 1:
        waveform = waveform.unsqueeze(0).unsqueeze(0)
    # Si es un tensor 2D (puede ser [channels, samples] o [samples, channels])
    elif len(waveform.shape) == 2:
        # Asumimos que es [samples, channels] y lo transponemos si es necesario
        if waveform.shape[1] > 2:  # Probablemente [samples, channels]
            waveform = waveform.T
        waveform = waveform.unsqueeze(0)  # Añadir dimensión de batch

    # Verificar que ahora tiene la forma correcta [1, channels, samples]
    print(f"Forma transformada: {waveform.shape}")

    # Asegurarse de que solo haya 1 canal
    if waveform.shape[1] > 1:
        # Si hay más de un canal, convertir a mono promediando los canales
        waveform = torch.mean(waveform, dim=1, keepdim=True)
        print(f"Forma después de convertir a mono: {waveform.shape}")

    # Verificación final
    assert waveform.shape[0] == 1, "Batch size should be 1"
    assert waveform.shape[1] == 1, "Number of channels should be 1"

    # Ahora debería estar en la forma [1, 1, samples]
    return embedding_model(waveform)

# Y luego usarla en tu código original
embeddings = np.zeros(shape=(len(segments), 192))
for i, segment in enumerate(segments):
  embeddings[i] = segment_embedding(segment)

embeddings = np.nan_to_num(embeddings)

Forma original: torch.Size([2, 583884])
Forma transformada: torch.Size([1, 583884, 2])
Forma después de convertir a mono: torch.Size([1, 1, 2])
Forma original: torch.Size([2, 233730])
Forma transformada: torch.Size([1, 233730, 2])
Forma después de convertir a mono: torch.Size([1, 1, 2])
Forma original: torch.Size([2, 92721])
Forma transformada: torch.Size([1, 92721, 2])
Forma después de convertir a mono: torch.Size([1, 1, 2])


In [21]:
clustering = AgglomerativeClustering(num_speakers).fit(embeddings)
labels = clustering.labels_
for i in range(len(segments)):
  segments[i]["speaker"] = 'SPEAKER ' + str(labels[i] + 1)

In [22]:
def time(secs):
  return datetime.timedelta(seconds=round(secs))

f = open("transcript.txt", "w")

for (i, segment) in enumerate(segments):
  if i == 0 or segments[i - 1]["speaker"] != segment["speaker"]:
    f.write("\n" + segment["speaker"] + ' ' + str(time(segment["start"])) + '\n')
  f.write(segment["text"][1:] + ' ')
f.close()